In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [5]:
from langchain_openrouter import ChatOpenRouter
model = ChatOpenRouter(model="openai/gpt-5-nano")   # or anthropic/claude-3.5-haiku, etc.

In [6]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    # "gpt-5-nano",
    model=model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [7]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [8]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='2e18e1d4-de14-4ea2-83a3-a26839508fe7'),
              AIMessage(content='', additional_kwargs={'reasoning_content': '**Updating favourite colour**\n\nThe user mentioned, "My favourite colour is green." It’s clear I need to update the user\'s favourite colour in my records. I’ll call the update_favourite_colour tool with their revealed colour, which is "green." After that, I want to confirm the update with a friendly note, perhaps saying something like, "Nice! I\'ve saved green as your favourite colour." I might also ask if they want to do anything else or keep using it in future features. I\'ll keep the formatting minimal.**Using the tool for updates**\n\nI need to use the appropriate tool to update the user\'s favourite colour, which is "green." I’ll call the functions.update_favourite_colour to do this. After that, I’ll respond with somethin

In [9]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='4c0c6810-b51e-42ca-9ea0-50b3e537fded'),
              AIMessage(content="Hi there! I'm here and ready to help. How are you today? What would you like to do—answer questions, brainstorm ideas, or assist with a task?", additional_kwargs={'reasoning_content': '**Crafting a friendly response**\n\nI need to respond to the user, who said, "Hello, how are you?" It sounds like they want a friendly exchange. I’ll reply with something like, "I\'m here and ready to help! How can I assist you today?" I might also ask how they’re doing, but I want to keep it concise. I should mention my image input capabilities and that I can assist with various tasks, like answering questions or brainstorming ideas.**Creating a welcoming message**\n\nI want to invite the user to share what they\'re curious about or what task they want to work on. A friendly approach could be sayin

## Read state

In [11]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    # "gpt-5-nano",
    model=model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [12]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='8ebf980b-96e2-4440-ba7e-400ad3068024'),
              AIMessage(content='', additional_kwargs={'reasoning_content': '**Updating favourite colour**\n\nI need to respond to the user who said, "My favourite colour is green." It seems like they want me to update their favourite colour in the system. I can use the tool to set their favourite colour as "green." Since the user mentioned it in lowercase, I should probably store it as "green" to keep it consistent. I\'ll confirm by saying, "Got it! I\'ve saved your favourite colour as green." I could also ask if they want to read it back!**Processing favourite colour update**\n\nIt looks like I only need to use the `update_favourite_colour` tool after the user reveals their favourite colour. So, I\'ll call it with the input "green" to update the state. Then, I should confirm by responding, "Thanks! I\'

In [13]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='8ebf980b-96e2-4440-ba7e-400ad3068024'),
              AIMessage(content='', additional_kwargs={'reasoning_content': '**Updating favourite colour**\n\nI need to respond to the user who said, "My favourite colour is green." It seems like they want me to update their favourite colour in the system. I can use the tool to set their favourite colour as "green." Since the user mentioned it in lowercase, I should probably store it as "green" to keep it consistent. I\'ll confirm by saying, "Got it! I\'ve saved your favourite colour as green." I could also ask if they want to read it back!**Processing favourite colour update**\n\nIt looks like I only need to use the `update_favourite_colour` tool after the user reveals their favourite colour. So, I\'ll call it with the input "green" to update the state. Then, I should confirm by responding, "Thanks! I\'